# GENCO inference on a GPU: how does it compare with the Newton break-even?

**Why this notebook exists.** On an Apple M4 the released GENCO power-flow model needs 7.7 to 24 ms per scenario (IEEE 118, CPU and MPS),
while the cheapest classical solver (fast-decoupled load flow from a base-case start) finishes a whole contingency in about 0.35 ms. A GENCO start only pays off in wall-clock time if
inference costs less than the solver time it saves: about 0.31 ms per contingency against a full Newton-Raphson, 0.04 ms against a
compensated chord solver and 0.015 ms against fast-decoupled load flow (IEEE 118 paper set; N-5: 0.63 / 0.06 / 0.04 ms).
A GPU with large batches is the most plausible way for GENCO inference to get near those numbers. This notebook measures it.

**What it measures** (forward pass only, data already in GPU memory, `torch.cuda.synchronize` around every timed call, warm-up, median of at least 10 repeats):

* GENCO (released seed-0 checkpoints) on IEEE 118 (Base, Small, Tiny), GOC 500 (Base, Small, Tiny) and GOC 2000 (Small, Tiny);
* batch sizes 1, 16, 64, 256, 1024 (batches above the memory cap are skipped, scenarios are repeated cyclically to reach the batch size);
* fp32 and bf16 autocast (plus fp16 on pre-Ampere GPUs such as the T4, TF32 on A100/L4/H100);
* the full 12-layer model and the 9-layer early exit;
* host-to-device transfer of the batch, reported separately;
* optional: one batched dense Newton step (`torch.linalg.solve` on stacked IEEE 118 Jacobians, float64 and float32) against a sparse LU on the host CPU;
* a precision check: how far bf16/fp16/TF32 predictions move from the fp32 predictions.

**Output.** A tidy CSV (`genco_gpu_timing.csv`: grid, size, device, dtype, batch, layers, ms_per_scenario, ...), a markdown report with the
break-even comparison, a figure and `env.json`, all zipped for download.

**How to run it**

1. *Runtime > Change runtime type > T4 GPU* (the free tier) or an A100/L4 if you have one. The notebook stops with an error if there is no GPU.
2. On your own machine run `./scripts/pack_for_colab.sh` in the repository root. It writes `dist/genco_colab_bundle.tar.gz` (about 0.2 MB).
   Our GitHub repository is private, so this tarball is how the code gets to Colab.
3. Run the cells in order. The upload cell opens a file picker; choose the tarball. If the install cell prints `RESTART REQUIRED`, use
   *Runtime > Restart session*, re-run the parameters cell and continue from the environment cell.
4. Last cell: the results zip downloads (or take it from the Files panel).

**Expected runtime:** about 5 to 10 minutes for install and data, then 20 to 45 minutes of timing on a T4 (roughly half on an A100).
Set `DRY_RUN = True` for a quick check of the notebook itself once the install is done (one model, tiny batches, one repeat, one data partition; the numbers it produces are meaningless).
Every cell can be re-run; finished models are kept in `results/gpu_timing/partial` and skipped unless `FORCE = True`.

In [ ]:
# ---------------- parameters: edit here, then run this cell (re-run it after a session restart) ----------------
import os

DRY_RUN = False        # True: smoke test of the notebook code (1 model, tiny batches, 1 repeat). Results are not meaningful.
FORCE = False          # True: redo models whose partial results already exist
FORCE_INSTALL = False  # True: run the pip steps even if graphkit, torch_geometric and our package are already installed

# grid -> HF partitions to download (200 scenarios each), model sizes to time, scenarios kept in the pool that batches are built from
GRIDS = {
    "case118_ieee": dict(partitions=5, sizes=["base", "small", "tiny"], pool=128),
    "case500_goc":  dict(partitions=5, sizes=["base", "small", "tiny"], pool=64),
    "case2000_goc": dict(partitions=2, sizes=["small", "tiny"],         pool=32),
}
BATCH_SIZES = [1, 16, 64, 256, 1024]
MAX_BUSES_PER_BATCH = 1_000_000   # skip batches with more buses than this (memory cap; GOC 2000 -> at most 256 scenarios; raise it on a big GPU)
DTYPES = "auto"                   # "auto" = fp32, bf16, and fp16 (pre-Ampere) or tf32 (Ampere and newer); or an explicit list of "fp32" "tf32" "bf16" "fp16"
EARLY_EXIT = 9                    # also time the model truncated to this many layers (None to skip)
WARMUP = 3
REPEATS = 15                      # timed repeats per configuration
MIN_REPEATS = 10                  # never fewer than this, even for slow configurations
TIME_BUDGET_S = 60                # soft cap per configuration: repeats are reduced (down to MIN_REPEATS) if REPEATS would take longer
RUN_NEWTON = True                 # optional batched dense Newton step on the GPU (IEEE 118)
NEWTON_GRID, NEWTON_SIZE, NEWTON_POOL = "case118_ieee", "base", 64
NEWTON_BATCH_SIZES = [1, 16, 64, 256]
MAX_SCENARIOS = None              # None: the splits cover all downloaded scenarios ("test = all")

TARBALL = "/content/genco_colab_bundle.tar.gz"                       # where the uploaded bundle is stored
REPO = os.environ.get("GENCO_CERTIFIED_ROOT", "/content/genco")      # where it is unpacked

if DRY_RUN:
    GRIDS = {"case118_ieee": dict(partitions=1, sizes=["base"], pool=8)}   # one partition is enough (more on disk is fine)
    BATCH_SIZES, NEWTON_BATCH_SIZES = [1, 4], [1, 4]
    DTYPES = ["fp32", "bf16"]
    WARMUP, REPEATS, MIN_REPEATS, TIME_BUDGET_S = 1, 1, 1, 5
    NEWTON_POOL, MAX_SCENARIOS = 6, 16

In [ ]:
import shutil, subprocess

smi = shutil.which("nvidia-smi")
if smi:
    print(subprocess.run([smi, "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True).stdout)
elif DRY_RUN:
    print("no nvidia-smi (fine for DRY_RUN)")
else:
    raise RuntimeError("No GPU found. Use Runtime > Change runtime type > T4 GPU (or better), then run this cell again.")

## 1. Upload our code

On your own machine, in the repository root, run `./scripts/pack_for_colab.sh`. It writes `dist/genco_colab_bundle.tar.gz` (source, scripts, configs, a few tests and
the tiny split files; no data, checkpoints or results). Run the next cell and pick that file. Alternatively copy it to Google Drive, mount Drive
(`from google.colab import drive; drive.mount("/content/drive")`) and set `TARBALL = "/content/drive/MyDrive/genco_colab_bundle.tar.gz"` in the parameters cell.

In [ ]:
import shutil, tarfile
from pathlib import Path

if not (Path(REPO) / "src/genco_certified/__init__.py").exists():
    if not Path(TARBALL).exists():
        try:
            from google.colab import files          # opens a file picker
        except ImportError:
            raise FileNotFoundError(f"{TARBALL} not found and this is not Colab: unpack the bundle into {REPO} yourself")
        up = files.upload()
        assert up, "no file chosen"
        shutil.move(next(iter(up)), TARBALL)        # whatever the file was called
    Path(REPO).mkdir(parents=True, exist_ok=True)
    with tarfile.open(TARBALL) as tf:
        try:
            tf.extractall(REPO, filter="data")
        except TypeError:                           # Python < 3.12
            tf.extractall(REPO)
    hits = list(Path(REPO).rglob("src/genco_certified/__init__.py"))
    assert hits, "the tarball must contain src/genco_certified (build it with scripts/pack_for_colab.sh)"
    top = hits[0].parents[2]                        # tolerate a wrapper folder inside the tarball
    if top != Path(REPO):
        for item in top.iterdir():
            shutil.move(str(item), REPO)

need = ["src/genco_certified/loading.py", "src/genco_certified/acpf.py", "src/genco_certified/graphkit_plugin.py", "pyproject.toml"]
missing = [f for f in need if not (Path(REPO) / f).exists()]
assert not missing, f"missing in the tarball: {missing}"
info = Path(REPO) / "BUNDLE_INFO.txt"
print("repository at", REPO)
print(info.read_text() if info.exists() else "(no BUNDLE_INFO.txt)")

## 2. Install dependencies

Colab ships `torch` (with CUDA). We keep it, look up its version and CUDA tag, and install the matching `torch-scatter` wheel from
`https://data.pyg.org/whl/torch-<version>+cu<cuda>.html` (PyG publishes wheels for `X.Y.0` that also work for later patch releases, so both spellings are tried).
If no wheel exists, `gridfm-graphkit` still needs the three `torch_scatter` functions it uses, so a small pure-PyTorch stand-in is written to
`/content/shims`; **the environment cell records which backend was used** (scatter-add is a small part of the forward pass, but it is not zero).
Then: `torch_geometric`, `gridfm-graphkit` at the commit our results were produced with (installed without dependency resolution, so pip does not touch torch),
`pandas<3`, scipy, pyarrow, huggingface_hub and the packages graphkit imports at module level; finally our own package (`pip install -e`).

If the cell reports `RESTART REQUIRED` (pip replaced numpy or pandas that were already imported): *Runtime > Restart session*, re-run the parameters cell, skip the upload and install cells.

In [ ]:
import importlib.metadata as md, os, shutil, subprocess, sys
from pathlib import Path

GRAPHKIT = "gridfm-graphkit @ git+https://github.com/gridfm/gridfm-graphkit@8a8defbc9ab41caa65b07c75e8e1ee23636877cd"
SHIM = Path("/content/shims")

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    rc = subprocess.run(cmd, shell=True).returncode
    if check and rc:
        raise RuntimeError(f"command failed ({rc}): {cmd}")
    return rc

def ver(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

def can_import(mod):
    env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}      # ignore the stand-in directory when probing for the real thing
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True, env=env).returncode == 0

SHIM_SRC = """
# Pure-PyTorch stand-in for the three torch_scatter functions gridfm-graphkit uses.
import torch

def _size(src, index, dim, dim_size):
    if dim_size is None:
        dim_size = int(index.max()) + 1 if index.numel() else 0
    size = list(src.shape)
    size[dim] = dim_size
    return size

def _expand(src, index, dim):
    if index.dim() == 1 and src.dim() > 1:
        shape = [1] * src.dim()
        shape[dim] = -1
        index = index.view(shape).expand_as(src)
    return index

def scatter_add(src, index, dim=-1, out=None, dim_size=None):
    dim = dim % src.dim()
    if out is None:
        out = torch.zeros(_size(src, index, dim, dim_size), dtype=src.dtype, device=src.device)
    if index.dim() == 1:
        return out.index_add_(dim, index, src)
    return out.scatter_add_(dim, _expand(src, index, dim), src)

def scatter_mean(src, index, dim=-1, out=None, dim_size=None):
    total = scatter_add(src, index, dim, None, dim_size)
    count = scatter_add(torch.ones_like(src), index, dim, None, dim_size)
    return total / count.clamp(min=1)

def scatter_max(src, index, dim=-1, out=None, dim_size=None):
    dim = dim % src.dim()
    low = torch.finfo(src.dtype).min
    init = torch.full(_size(src, index, dim, dim_size), low, dtype=src.dtype, device=src.device)
    vals = init.scatter_reduce(dim, _expand(src, index, dim), src, reduce="amax", include_self=True)
    vals = torch.where(vals == low, torch.zeros_like(vals), vals)
    return vals, torch.zeros_like(vals, dtype=torch.long)
"""

def install():
    before = {p: ver(p) for p in ("torch", "numpy", "pandas", "scipy")}
    if before["torch"] is None:
        raise RuntimeError("torch is not installed in this runtime")
    tv, cu = subprocess.run([sys.executable, "-c", "import torch;print(torch.__version__.split('+')[0], torch.version.cuda or 'cpu')"],
                            capture_output=True, text=True).stdout.split()
    print(f"preinstalled torch {tv}, CUDA {cu}")
    if cu == "cpu":
        raise RuntimeError("this torch build has no CUDA: Runtime > Change runtime type > GPU, then reconnect")

    # 1. torch-scatter: matching wheel from data.pyg.org, else the pure-torch stand-in
    have = can_import("torch_scatter") and not (SHIM / "torch_scatter").exists()
    if not have:
        tag = "cu" + cu.replace(".", "")
        major_minor = ".".join(tv.split(".")[:2])
        for t in dict.fromkeys([f"{major_minor}.0", tv]):          # PyG labels wheels X.Y.0
            sh(f"pip install -q --only-binary=:all: torch-scatter -f https://data.pyg.org/whl/torch-{t}+{tag}.html", check=False)
            have = can_import("torch_scatter")
            if have:
                break
        if have and (SHIM / "torch_scatter").exists():
            shutil.rmtree(SHIM / "torch_scatter")                  # a real wheel is available now
    if not have:
        (SHIM / "torch_scatter").mkdir(parents=True, exist_ok=True)
        (SHIM / "torch_scatter/__init__.py").write_text(SHIM_SRC)
        print(f"torch_scatter: no wheel for torch {tv}+{cu}; using the pure-PyTorch stand-in in {SHIM} (recorded in env.json)")

    # 2. gridfm-graphkit (no dependency resolution against torch) and the packages it needs at import time
    if ver("gridfm-graphkit") is None:
        sh(f'pip install -q --no-deps --ignore-requires-python "{GRAPHKIT}"')
    sh('pip install -q --ignore-requires-python "pandas>=2.3,<3" "numpy>=2.2.6" scipy pyarrow huggingface_hub '
       '"mlflow>=3.1.0" nbformat networkx opt-einsum plotly pyyaml "torch-geometric>=2.6.1" lightning seaborn '
       '"urllib3>=2.6.0" "gdown>=6.0.0" "gridfm-datakit==1.1.0"')

    # 3. our package from the unpacked bundle (no dependencies: they are installed above, and pip must not replace torch)
    sh(f'pip install -q --no-deps --ignore-requires-python -e "{REPO}"')

    after = {p: ver(p) for p in before}
    changed = {p: (before[p], after[p]) for p in before if before[p] is not None and before[p] != after[p]}
    print("\ninstalled:", {**after, "torch-geometric": ver("torch-geometric"), "gridfm-graphkit": ver("gridfm-graphkit")})
    if changed:
        print("\n" + "=" * 78)
        print("RESTART REQUIRED: pip changed", changed)
        print("Runtime > Restart session, then re-run the parameters cell and continue with the environment cell.")
        print("=" * 78)
    else:
        print("no restart needed")

def installed():
    needed = ["gridfm-graphkit", "torch-geometric", "gridfm-datakit", "genco-certified"]
    return all(ver(p) for p in needed) and (can_import("torch_scatter") or (SHIM / "torch_scatter").exists())

if installed() and not FORCE_INSTALL:
    print("graphkit, torch_geometric, datakit and genco-certified are already installed: nothing to do (FORCE_INSTALL = True runs pip again)")
else:
    install()

## 3. Environment check

Points Python at the unpacked repository, selects the GPU (and **fails if there is none**), prints the versions that matter for the timings and writes `env.json`.

In [ ]:
import os, sys, json, time, copy, gc, math, platform, hashlib, contextlib, zipfile, traceback
import importlib.metadata as md
from pathlib import Path

SHIM = "/content/shims"
paths = [f"{REPO}/src"] + ([SHIM] if Path(SHIM, "torch_scatter").exists() else [])
os.environ["PYTHONPATH"] = os.pathsep.join(paths + [os.environ.get("PYTHONPATH", "")]).rstrip(os.pathsep)
for p in paths[::-1]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ.update(GENCO_CERTIFIED_ROOT=REPO, MLFLOW_ALLOW_FILE_STORE="true", HF_HUB_DISABLE_PROGRESS_BARS="1")
os.chdir(REPO)

import numpy as np
import pandas as pd
import torch

CUDA = torch.cuda.is_available()
if not CUDA and not DRY_RUN:
    raise RuntimeError("CUDA is not available to PyTorch: Runtime > Change runtime type > GPU (and reconnect). Refusing to produce CPU timings.")
DEVICE = torch.device("cuda" if CUDA else "cpu")
if DRY_RUN and not CUDA:
    torch.set_num_threads(2)
DEVICE_NAME = torch.cuda.get_device_name(0) if CUDA else "cpu"
CC = torch.cuda.get_device_capability(0) if CUDA else (0, 0)
if DTYPES == "auto":
    DTYPES = ["fp32", "bf16"] + (["tf32"] if CC >= (8, 0) else ["fp16"] if CUDA else [])
DTYPES = list(DTYPES)
assert set(DTYPES) <= {"fp32", "tf32", "bf16", "fp16"}, DTYPES

import genco_certified.loading as GL
assert Path(GL.ROOT).resolve() == Path(REPO).resolve(), f"genco_certified.loading was imported with ROOT={GL.ROOT}; restart the runtime"
from genco_certified.loading import load_genco
from genco_certified import acpf

try:
    import torch_scatter
    SCATTER = "pure-torch stand-in" if "shims" in str(torch_scatter.__file__) else "torch_scatter " + torch_scatter.__version__
except Exception as e:                      # pragma: no cover
    SCATTER = f"unavailable ({e})"

def _cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()

def _ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return None

ENV = dict(
    date_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), dry_run=DRY_RUN, device=DEVICE_NAME, device_type=DEVICE.type,
    compute_capability=f"{CC[0]}.{CC[1]}" if CUDA else None,
    gpu_memory_gb=round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if CUDA else None,
    bf16_native=bool(CUDA and CC >= (8, 0)),
    python=sys.version.split()[0], torch=torch.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
    torch_geometric=_ver("torch-geometric"), gridfm_graphkit=_ver("gridfm-graphkit"), scatter_backend=SCATTER,
    numpy=np.__version__, pandas=pd.__version__, scipy=_ver("scipy"),
    host_cpu=_cpu_model(), host_cpus=os.cpu_count(), torch_threads=torch.get_num_threads(),
    float32_matmul_precision=torch.get_float32_matmul_precision(), dtypes=DTYPES, batch_sizes=BATCH_SIZES, early_exit=EARLY_EXIT,
    warmup=WARMUP, repeats=REPEATS, min_repeats=MIN_REPEATS, max_buses_per_batch=MAX_BUSES_PER_BATCH,
    bundle=(Path(REPO) / "BUNDLE_INFO.txt").read_text() if (Path(REPO) / "BUNDLE_INFO.txt").exists() else None,
)
print(json.dumps({k: v for k, v in ENV.items() if k != "bundle"}, indent=1))
if CUDA and CC < (8, 0):
    print("\nnote: this GPU has no native bf16 (compute capability < 8.0); bf16 autocast may be emulated or refused. fp16 is the fast reduced precision here.")

OUT = Path(REPO) / "results" / ("gpu_timing_dry" if DRY_RUN else "gpu_timing")
PARTIAL = OUT / "partial"
PARTIAL.mkdir(parents=True, exist_ok=True)
(OUT / "env.json").write_text(json.dumps(ENV, indent=1))

## 4. Data

The first partitions (200 scenarios each) of the public `gridfm/pf_small_<grid>` datasets are downloaded into `data/pf/<grid>/raw`
(graphkit's streaming loader needs scenario ids `0..N-1`, hence a contiguous prefix). Only the forward pass is timed, so any scenarios will do:
the split is simply **test = all downloaded scenarios** (train and val are two dummy ids because the loader insists on non-empty files).
GOC 500 and GOC 2000 have fewer scenarios than the largest batch sizes need; batches are then built by repeating the pool cyclically (the forward pass does not depend on the values).

In [ ]:
import torch
from huggingface_hub import snapshot_download

TABLES = ["bus_data", "gen_data", "branch_data"]
DATA = Path(REPO) / "data" / "pf"
SPLITS = {}

def fetch(grid, n_partitions):
    raw = DATA / grid / "raw"
    tbl = raw / "bus_data.parquet"
    have = len(list(tbl.glob("scenario_partition=*"))) if tbl.exists() else 0
    if n_partitions and (FORCE or have < n_partitions):
        patterns = [f"{t}.parquet/scenario_partition={i}/*" for t in TABLES for i in range(n_partitions)]
        snapshot_download(f"gridfm/pf_small_{grid}", repo_type="dataset", local_dir=raw, allow_patterns=patterns + ["n_scenarios.txt", "args.log"])
    elif not tbl.exists():
        raise FileNotFoundError(f"{tbl} not found")
    ids = np.sort(pd.read_parquet(tbl, columns=["scenario"])["scenario"].unique())
    assert ids[0] == 0 and ids[-1] == len(ids) - 1, f"{grid}: scenario ids are not contiguous 0..N-1 ({ids[0]}..{ids[-1]}, {len(ids)} ids)"
    return int(len(ids))

def write_split(grid, n_total):
    n = n_total if MAX_SCENARIOS is None else min(n_total, MAX_SCENARIOS)
    d = (OUT / "dry_splits" / grid) if DRY_RUN else (DATA / grid / "splits_timing")   # dry run: do not touch the project's data folders
    d.mkdir(parents=True, exist_ok=True)
    torch.save(torch.arange(0, min(2, n), dtype=torch.long), d / "train.pt")
    torch.save(torch.arange(min(2, n), min(4, n), dtype=torch.long) if n > 2 else torch.arange(0, 1), d / "val.pt")
    torch.save(torch.arange(0, n, dtype=torch.long), d / "test.pt")
    return d, n

for grid, cfg in GRIDS.items():
    n_total = fetch(grid, cfg["partitions"])
    SPLITS[grid], n_used = write_split(grid, n_total)
    print(f"{grid}: {n_total} scenarios on disk, test split = first {n_used}  ({SPLITS[grid]})")

## 5. Timing helpers

Definitions only. `time_call` synchronises the GPU before and after every timed call, does the warm-up calls first and returns the median
(plus quartiles) of the repeats. A configuration that runs out of memory or raises (for example an unsupported autocast dtype) is recorded with its status
and the larger batch sizes of that configuration are skipped; the run continues.

In [ ]:
# Break-even GENCO inference time per contingency, IEEE 118, in ms: the solver time that a GENCO start saves compared with the base-case start
# (same solver, tolerance 1e-4 MW, "measured" basis), so GENCO inference must cost less than this to pay off in wall-clock time.
# Sources (column breakeven_same_solver_ms, tol = 0.0001): results/exp07/breakeven.csv (paper set, 638 contingencies) and
# results/exp07/breakeven_stress_n5_base.csv (N-5 stress set, 660 contingencies). The solver timings behind them were taken on an Apple M4,
# one thread, python/scipy implementation (RESEARCH_LOG.md, "Batch 2: industry-style baselines").
BREAKEVEN_MS = {
    "paper set (N-1)": {"full NR": 0.306, "compensated chord": 0.0368, "FDLF": 0.0150},
    "N-5 stress":      {"full NR": 0.630, "compensated chord": 0.0577, "FDLF": 0.0415},
}
# Sparse LU solve of the 181x181 IEEE 118 Jacobian on the M4 (results/runtime/case118_ieee_base.json, lu_solve_ms); used to rescale the
# thresholds to the CPU of this runtime (they are CPU solver times).
M4_LU_SOLVE_MS = 0.0641
# GENCO forward pass on the M4, ms per scenario (results/runtime/case118_ieee_base.json and case2000_goc_small.json), for orientation only.
M4_GENCO_MS = {
    ("case118_ieee", "base"):  {"CPU, batch 16": 12.13, "MPS, batch 128": 10.31, "MPS, batch 128, 9 layers": 7.67},
    ("case2000_goc", "small"): {"CPU, batch 16": 102.78, "MPS, batch 64": 100.33, "MPS, batch 64, 9 layers": 71.04},
}

def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def free():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()

def early_exit(model, n_layers):
    """Copy of GENCO that stops after `n_layers` correction layers (the decoder heads are shared, as in scripts/bench_runtime.py)."""
    m = copy.deepcopy(model)
    m.layers = m.layers[:n_layers]
    m.num_layers = n_layers
    return m

@contextlib.contextmanager
def precision(name):
    if name == "fp32":
        yield
    elif name == "tf32":
        old = torch.get_float32_matmul_precision()
        torch.set_float32_matmul_precision("high")
        try:
            yield
        finally:
            torch.set_float32_matmul_precision(old)
    elif name in ("bf16", "fp16"):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16 if name == "bf16" else torch.float16):
            try:
                on = torch.is_autocast_enabled(DEVICE.type)
            except TypeError:                                   # older torch: cuda only
                on = torch.is_autocast_enabled()
            if not on:
                raise RuntimeError(f"autocast {name} is not enabled on {DEVICE_NAME}")
            yield
    else:
        raise ValueError(name)

def time_call(fn, reps=None, warmup=None, budget=None):
    """Median seconds of fn() (synchronised), with quartiles and the number of repeats actually used."""
    reps, warmup, budget = reps or REPEATS, WARMUP if warmup is None else warmup, budget or TIME_BUDGET_S
    last = 0.0
    for _ in range(warmup):
        t0 = time.perf_counter()
        fn()
        sync()
        last = time.perf_counter() - t0          # the last warm-up call is the best estimate of the steady-state time
    n = reps if last * reps <= budget else max(MIN_REPEATS, int(budget / max(last, 1e-9)))
    ts = []
    for _ in range(n):
        sync()
        t = time.perf_counter()
        fn()
        sync()
        ts.append(time.perf_counter() - t)
    return float(np.median(ts)), float(np.percentile(ts, 25)), float(np.percentile(ts, 75)), n

def make_batch(pool, bs):
    from torch_geometric.data import Batch
    return Batch.from_data_list([pool[i % len(pool)] for i in range(bs)])

def row(kind, grid, size, dtype, batch, layers, med=None, q25=None, q75=None, reps=0, mem=np.nan, status="ok"):
    per = (lambda x: 1e3 * x / batch) if med is not None else (lambda x: np.nan)
    return dict(kind=kind, grid=grid, size=size, device=DEVICE_NAME, dtype=dtype, batch=batch, layers=layers,
                ms_per_scenario=per(med), ms_per_scenario_q25=per(q25), ms_per_scenario_q75=per(q75),
                ms_batch=1e3 * med if med is not None else np.nan, reps=reps, peak_mem_mb=mem, status=status)

def measure_h2d(cpu_batch, grid, size, bs):
    """Host-to-device copy of one batch (pageable memory), median of REPEATS copies; the clone is made outside the timed region."""
    ts = []
    for i in range(WARMUP + max(REPEATS, MIN_REPEATS)):
        b = cpu_batch.clone()
        sync()
        t = time.perf_counter()
        b.to(DEVICE)
        sync()
        if i >= WARMUP:
            ts.append(time.perf_counter() - t)
        del b
    return row("h2d", grid, size, "-", bs, np.nan, float(np.median(ts)), float(np.percentile(ts, 25)), float(np.percentile(ts, 75)), len(ts))

def time_forward(model, dev_batch, grid, size, dtype, layers, bs):
    try:
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        with torch.no_grad(), precision(dtype):
            med, q25, q75, n = time_call(lambda: model(dev_batch))
        mem = torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else np.nan
        return row("forward", grid, size, dtype, bs, layers, med, q25, q75, n, mem), None
    except Exception as e:
        oom, msg = is_oom(e), f"{type(e).__name__}: {str(e)[:100]}"
    free()                                   # outside the except block, so the traceback no longer pins GPU tensors
    if oom:
        return row("forward", grid, size, dtype, bs, layers, status="OOM"), "OOM"
    print(f"    {grid}/{size} {dtype} {layers} layers batch {bs}: {msg}")
    return row("forward", grid, size, dtype, bs, layers, status=f"error: {msg}"), "error"

def precision_check(model, items, grid, size, layers):
    """Largest and median deviation of the bus predictions (Vm in p.u., Va in rad, normalised units) from the fp32 result."""
    batch = make_batch(items, min(16, len(items))).to(DEVICE)
    out = []
    with torch.no_grad():
        ref = model(batch)["bus"].float()
        for dtype in DTYPES:
            if dtype == "fp32":
                continue
            try:
                with precision(dtype):
                    y = model(batch)["bus"].float()
                d = (y - ref).abs()
                out.append(dict(grid=grid, size=size, device=DEVICE_NAME, dtype=dtype, layers=layers,
                                max_abs_dev_vm=d[:, 0].max().item(), max_abs_dev_va_rad=d[:, 1].max().item(),
                                median_abs_dev_vm=d[:, 0].median().item(), median_abs_dev_va_rad=d[:, 1].median().item(),
                                nonfinite=int((~torch.isfinite(y)).sum().item()), status="ok"))
            except Exception as e:
                out.append(dict(grid=grid, size=size, device=DEVICE_NAME, dtype=dtype, layers=layers, status=f"error: {type(e).__name__}: {str(e)[:100]}"))
    del batch
    free()
    return out

def run_model(grid, size, cfg):
    L = load_genco("pf", grid, size, seed=0, split_dir=SPLITS[grid], batch_size=min(cfg["pool"], 32))
    items = []
    for b in L.test_loader():
        items += b.to_data_list()
        if len(items) >= cfg["pool"]:
            break
    items = items[: cfg["pool"]]
    n_bus = int(items[0]["bus"].x.shape[0])
    model = L.task.model.eval().to(DEVICE)
    n_layers = int(model.num_layers)
    variants = {n_layers: model}
    if EARLY_EXIT and EARLY_EXIT < n_layers:
        variants[EARLY_EXIT] = early_exit(model, EARLY_EXIT)
    print(f"{grid}/{size}: {n_bus} buses, {sum(p.numel() for p in model.parameters()):,} parameters, {len(items)} pool scenarios, layers {list(variants)}")

    rows, prec, failed = [], [], {}
    for layers, m in variants.items():
        prec += precision_check(m, items, grid, size, layers)
    for bs in BATCH_SIZES:
        if bs * n_bus > MAX_BUSES_PER_BATCH:
            rows.append(row("forward", grid, size, "-", bs, n_layers, status=f"skipped: {bs} x {n_bus} buses > MAX_BUSES_PER_BATCH"))
            continue
        cpu_batch = make_batch(items, bs)
        try:
            rows.append(measure_h2d(cpu_batch, grid, size, bs))
            dev_batch = cpu_batch.to(DEVICE)               # in place: cpu_batch is the same object afterwards
        except Exception as e:
            status = "OOM" if is_oom(e) else f"error: {type(e).__name__}: {str(e)[:100]}"
            rows.append(row("h2d", grid, size, "-", bs, np.nan, status=status))
            cpu_batch = None
            free()
            continue
        for layers, m in variants.items():
            for dtype in DTYPES:
                if (dtype, layers) in failed:
                    continue
                r, flag = time_forward(m, dev_batch, grid, size, dtype, layers, bs)
                rows.append(r)
                if flag:
                    failed[(dtype, layers)] = flag
                if r["status"] == "ok":
                    print(f"    batch {bs:>5} {dtype:>5} {layers:>2} layers: {r['ms_per_scenario']:9.4f} ms/scenario   ({r['reps']} reps, {r['peak_mem_mb']:.0f} MB peak)", flush=True)
                else:
                    print(f"    batch {bs:>5} {dtype:>5} {layers:>2} layers: {r['status']}", flush=True)
        del dev_batch, cpu_batch
        free()
    del model, variants, L, items
    free()
    return rows, prec

## 6. Time GENCO

For each model: precision check, then for each batch size the host-to-device copy and the forward pass for every dtype and layer count. Each finished model is written to
`results/gpu_timing/partial`, so after a disconnect you can re-run this cell and it continues with the models that are missing.

In [ ]:
SIG = hashlib.md5(json.dumps([DEVICE_NAME, BATCH_SIZES, DTYPES, EARLY_EXIT, REPEATS, MIN_REPEATS, WARMUP, MAX_BUSES_PER_BATCH, MAX_SCENARIOS, DRY_RUN]).encode()).hexdigest()[:6]
slug = "".join(c if c.isalnum() else "_" for c in DEVICE_NAME).strip("_")
ALL_ROWS, ALL_PREC, FAILED_MODELS = [], [], []
t_start = time.time()
for grid, cfg in GRIDS.items():
    for size in cfg["sizes"]:
        f_t, f_p = PARTIAL / f"{grid}_{size}_{slug}_{SIG}_timing.csv", PARTIAL / f"{grid}_{size}_{slug}_{SIG}_precision.csv"
        if f_t.exists() and f_p.exists() and not FORCE:
            print(f"skip {grid}/{size} (partial results exist; FORCE = True to redo)")
            ALL_ROWS.append(pd.read_csv(f_t))
            ALL_PREC.append(pd.read_csv(f_p))
            continue
        try:
            rows, prec = run_model(grid, size, cfg)
        except Exception:
            print(f"!! {grid}/{size} failed:\n{traceback.format_exc()}")
            FAILED_MODELS.append(f"{grid}/{size}")
            free()
            continue
        pd.DataFrame(rows).to_csv(f_t, index=False)
        pd.DataFrame(prec).to_csv(f_p, index=False)
        ALL_ROWS.append(pd.DataFrame(rows))
        ALL_PREC.append(pd.DataFrame(prec))
        print(f"   elapsed {time.time() - t_start:.0f}s", flush=True)
assert ALL_ROWS, "no model could be timed; see the errors above"
if FAILED_MODELS:
    print("\nmodels that failed (re-run this cell after fixing the cause):", FAILED_MODELS)
DF = pd.concat(ALL_ROWS, ignore_index=True)
DF_PREC = pd.concat(ALL_PREC, ignore_index=True) if ALL_PREC else pd.DataFrame()
print(f"\n{len(DF)} timing rows, {int((DF.status == 'ok').sum())} ok, in {time.time() - t_start:.0f}s")

## 7. Optional: one batched dense Newton step on the GPU (IEEE 118)

A different kind of GPU other entries for GENCO: build the IEEE 118 Jacobians at the flat start on the CPU (`genco_certified.acpf`), stack them and solve all Newton
steps at once with `torch.linalg.solve` in float64 and float32. This times only the linear solve, not the Jacobian assembly or the extra iterations Newton needs,
so it bounds one Newton step from below. The reference is the sparse LU (scipy) of one Jacobian on the host CPU, which also calibrates the CPU speed of this runtime against the M4
(the break-even thresholds are CPU solver times). Set `RUN_NEWTON = False` to skip.

In [ ]:
import scipy.sparse.linalg as spla
from collections import Counter

NEWTON_ROWS = []
HOST_LU_MS = None
if RUN_NEWTON:
    try:
        Ln = load_genco("pf", NEWTON_GRID, NEWTON_SIZE, seed=0, split_dir=SPLITS[NEWTON_GRID], batch_size=NEWTON_POOL)
        raw = next(iter(Ln.test_loader())).clone()
        Ln.normalizer.inverse_transform(raw)                         # physical units, as scripts/bench_runtime.py does
        probs = acpf.extract_problems(raw)[:NEWTON_POOL]
        Js, Fs, Jsp = [], [], []
        for p in probs:
            V = p.flat_start()
            J = acpf.jacobian(p, V)
            Jsp.append((J.tocsc(), acpf.mismatch(p, V)))
            Js.append(J.toarray())
            Fs.append(acpf.mismatch(p, V))
        n = Counter(J.shape[0] for J in Js).most_common(1)[0][0]
        keep = [i for i, J in enumerate(Js) if J.shape[0] == n]
        Js, Fs, Jsp = [Js[i] for i in keep], [Fs[i] for i in keep], [Jsp[i] for i in keep]
        print(f"{len(Js)} of {len(probs)} problems have the modal size n = {n} unknowns")

        # host reference: scipy sparse LU per scenario
        lu = []
        for J, F in Jsp:
            ts = []
            for _ in range(max(REPEATS, 3)):
                t = time.perf_counter(); spla.spsolve(J, -F); ts.append(time.perf_counter() - t)
            lu.append(float(np.median(ts)))
        HOST_LU_MS = 1e3 * float(np.median(lu))
        NEWTON_ROWS.append(dict(kind="newton_solve", grid=NEWTON_GRID, size="-", device="host CPU", dtype="float64 sparse LU (scipy)", batch=1, layers=np.nan,
                                ms_per_scenario=HOST_LU_MS, ms_per_scenario_q25=1e3 * float(np.percentile(lu, 25)), ms_per_scenario_q75=1e3 * float(np.percentile(lu, 75)),
                                ms_batch=HOST_LU_MS, reps=len(lu), peak_mem_mb=np.nan, status="ok",
                                note=f"M4 reference {M4_LU_SOLVE_MS} ms -> this host is {HOST_LU_MS / M4_LU_SOLVE_MS:.1f}x the M4 time"))
        print(f"host CPU sparse LU: {HOST_LU_MS:.4f} ms per Jacobian (M4: {M4_LU_SOLVE_MS} ms) -> host/M4 = {HOST_LU_MS / M4_LU_SOLVE_MS:.2f}")

        J64 = torch.tensor(np.stack(Js), dtype=torch.float64)
        F64 = torch.tensor(np.stack(Fs), dtype=torch.float64)
        for dev in dict.fromkeys([DEVICE, torch.device("cpu")]):
            for dt in (torch.float64, torch.float32):
                for bs in NEWTON_BATCH_SIZES:
                    idx = torch.arange(bs) % len(Js)
                    try:
                        A, B = J64[idx].to(dev, dt), F64[idx].to(dev, dt).unsqueeze(-1)
                        med, q25, q75, reps = time_call(lambda: torch.linalg.solve(A, B), warmup=None if dev == DEVICE else 1)
                        x = torch.linalg.solve(A, B)
                        res = ((J64[idx].to(dev) @ x.double() - F64[idx].to(dev).unsqueeze(-1)).abs().amax((1, 2)) / F64[idx].to(dev).abs().amax(1)).max().item()
                        NEWTON_ROWS.append(dict(kind="newton_solve", grid=NEWTON_GRID, size="-", device=DEVICE_NAME if dev == DEVICE else "host CPU",
                                                dtype=str(dt).replace("torch.", ""), batch=bs, layers=np.nan, ms_per_scenario=1e3 * med / bs,
                                                ms_per_scenario_q25=1e3 * q25 / bs, ms_per_scenario_q75=1e3 * q75 / bs, ms_batch=1e3 * med, reps=reps,
                                                peak_mem_mb=np.nan, status="ok", note=f"n={n}; max relative residual {res:.1e}"))
                    except Exception as e:
                        free()
                        NEWTON_ROWS.append(dict(kind="newton_solve", grid=NEWTON_GRID, size="-", device=DEVICE_NAME if dev == DEVICE else "host CPU",
                                                dtype=str(dt).replace("torch.", ""), batch=bs, layers=np.nan, ms_per_scenario=np.nan, status="OOM" if is_oom(e) else f"error: {type(e).__name__}: {str(e)[:100]}"))
        del Ln, raw, probs
        free()
        print(pd.DataFrame(NEWTON_ROWS)[["device", "dtype", "batch", "ms_per_scenario", "status", "note"]].to_string(index=False))
    except Exception:
        print("optional Newton part failed (the rest of the notebook is unaffected):\n" + traceback.format_exc())
else:
    print("RUN_NEWTON = False: skipped")

## 8. Report

Writes `genco_gpu_timing.csv` (one row per measurement: `kind` is `forward`, `h2d` or `newton_solve`), `genco_precision_check.csv`, `report.md` and a figure.
The report compares ms per scenario with the break-even thresholds of the IEEE 118 contingency study. Read the ratio column as: GENCO inference time divided by the solver time it saves.
Below 1, GENCO would pay off in wall-clock time against that solver on the M4 timings; above 1 it would not.

In [ ]:
FULL = pd.concat([DF, pd.DataFrame(NEWTON_ROWS)], ignore_index=True) if NEWTON_ROWS else DF
cols = ["kind", "grid", "size", "device", "dtype", "batch", "layers", "ms_per_scenario", "ms_per_scenario_q25", "ms_per_scenario_q75", "ms_batch", "reps", "peak_mem_mb", "status", "note"]
FULL = FULL.reindex(columns=cols)
FULL.to_csv(OUT / "genco_gpu_timing.csv", index=False)
if len(DF_PREC):
    DF_PREC.to_csv(OUT / "genco_precision_check.csv", index=False)

def fmt(v):
    if isinstance(v, (float, np.floating)):
        return "-" if not np.isfinite(v) else (f"{v:.3g}" if abs(v) < 1000 else f"{v:,.0f}")
    return "-" if v is None else str(v)

def md_table(df):
    c = list(df.columns)
    lines = ["| " + " | ".join(c) + " |", "|" + "|".join("---" for _ in c) + "|"]
    lines += ["| " + " | ".join(fmt(r[k]) for k in c) + " |" for _, r in df.iterrows()]
    return "\n".join(lines)

SIZE_ORDER = {"base": 0, "small": 1, "tiny": 2}
fw = DF[(DF.kind == "forward") & (DF.status == "ok")].copy()
h2d = DF[(DF.kind == "h2d") & (DF.status == "ok")].set_index(["grid", "size", "batch"]).ms_per_scenario

def summary(grid, thresholds=None):
    out = []
    for (size, dtype, layers), g in fw[fw.grid == grid].groupby(["size", "dtype", "layers"]):
        best = g.loc[g.ms_per_scenario.idxmin()]
        b1 = g[g.batch == 1].ms_per_scenario
        r = {"model": size, "dtype": dtype, "layers": int(layers), "batch 1 (ms)": float(b1.iloc[0]) if len(b1) else np.nan,
             "best (ms/scenario)": best.ms_per_scenario, "at batch": int(best.batch),
             "best + h2d (ms)": best.ms_per_scenario + h2d.get((grid, size, int(best.batch)), np.nan)}
        if thresholds:
            for name, t in thresholds.items():
                r[f"best / {name}"] = best.ms_per_scenario / t
            r["below break-even for"] = ", ".join(n for n, t in thresholds.items() if best.ms_per_scenario <= t) or "none"
        out.append(r)
    df = pd.DataFrame(out)
    if len(df):
        df = df.sort_values(["model", "layers", "dtype"], key=lambda s: s.map(SIZE_ORDER).fillna(s) if s.name == "model" else s, ascending=[True, False, True])
    return df

lines = [f"# GENCO GPU timing: {DEVICE_NAME}", "",
         f"Run {ENV['date_utc']}{' (DRY RUN: numbers are not meaningful)' if DRY_RUN else ''}. torch {ENV['torch']} (CUDA {ENV['cuda']}), torch_geometric {ENV['torch_geometric']}, "
         f"scatter backend: {ENV['scatter_backend']}. Host CPU: {ENV['host_cpu']} ({ENV['host_cpus']} cores).", "",
         "Forward pass only, data resident on the device, synchronised, median of the repeats (see `reps` in the CSV). `ms/scenario` = batch time / batch size. "
         "`best` = the batch size with the lowest ms/scenario; `best + h2d` adds the host-to-device copy of that batch (pageable memory) per scenario.", ""]
if "case118_ieee" in fw.grid.values:
    for setname, thr in BREAKEVEN_MS.items():
        lines += [f"## IEEE 118 vs the break-even thresholds, {setname}", "",
                  "Thresholds (ms per contingency, M4 / scipy): " + ", ".join(f"{k} {v}" for k, v in thr.items()) + ". "
                  "A ratio below 1 means GENCO inference is cheaper than the solver time its start saves.", "",
                  md_table(summary("case118_ieee", thr)), ""]
    if HOST_LU_MS:
        f = HOST_LU_MS / M4_LU_SOLVE_MS
        lines += [f"CPU calibration: a sparse LU solve of the IEEE 118 Jacobian takes {HOST_LU_MS:.4f} ms on this runtime against {M4_LU_SOLVE_MS} ms on the M4 (factor {f:.2f}). "
                  f"The classical solvers would be about {f:.2f}x slower here as well, so the thresholds scale to roughly "
                  + ", ".join(f"{k} {v * f:.3g}" for k, v in BREAKEVEN_MS["paper set (N-1)"].items()) + " ms (paper set).", ""]
other = [g for g in GRIDS if g != "case118_ieee" and g in fw.grid.values]
for grid in other:
    lines += [f"## {grid}", "", "No break-even thresholds exist for this grid (the contingency study is IEEE 118 only).", "", md_table(summary(grid)), ""]
if NEWTON_ROWS:
    nr = pd.DataFrame(NEWTON_ROWS)
    lines += ["## Batched dense Newton step (IEEE 118, linear solve only)", "",
              "Optional comparison: `torch.linalg.solve` on stacked 181 x 181 Jacobians versus one sparse LU on the host CPU. The Jacobian assembly and the remaining Newton iterations are not included.", "",
              md_table(nr[["device", "dtype", "batch", "ms_per_scenario", "reps", "status", "note"]]), ""]
if len(DF_PREC):
    lines += ["## Precision check", "",
              "Largest and median absolute deviation of the predicted bus Vm (p.u.) and Va (rad) from the fp32 prediction of the same model, on 16 scenarios. "
              "For scale: GENCO's median state error against the datakit solution is about 0.05 (IEEE 118, Base).", "",
              md_table(DF_PREC.drop(columns=["device"], errors="ignore")), ""]
ref = [(k, v) for k, v in M4_GENCO_MS.items() if k[0] in fw.grid.values]
if ref:
    lines += ["## Orientation: the same forward pass on the Apple M4", "", "ms per scenario, from `results/runtime/*.json`.", ""]
    lines += [f"* {g}/{s}: " + ", ".join(f"{k} {v}" for k, v in d.items()) for (g, s), d in ref] + [""]
bad = FULL[(FULL.kind != "newton_solve") & (FULL.status != "ok")]
if len(bad):
    lines += ["## Configurations that did not run", "", md_table(bad[["grid", "size", "dtype", "batch", "layers", "status"]].drop_duplicates()), ""]
lines += ["## Caveats", "",
          "* The thresholds come from a Python/scipy implementation on one M4 thread; a compiled solver on a server CPU would be faster, so the real break-even is lower.",
          "* Early exit after 9 layers costs about +0.1 Newton iteration (RESEARCH_LOG), which shrinks the saving it is compared with.",
          "* Batch 1 is latency (kernel launches and Python overhead, so it depends on the host CPU as much as on the GPU); large batches measure throughput.",
          "* Scenarios are repeated cyclically when a batch is larger than the pool; the forward pass does not depend on the values.",
          "* Free-tier GPUs are shared and clocks vary between sessions: repeat the run before quoting a number.",
          "* Reduced precision is only useful if the prediction still works as a Newton start; the precision table shows the deviation, not the effect on Newton iterations."]
REPORT = "\n".join(lines)
(OUT / "report.md").write_text(REPORT)
try:
    from IPython.display import display, Markdown
    display(Markdown(REPORT))
except Exception:
    print(REPORT)

In [ ]:
import matplotlib
import matplotlib.ticker
import matplotlib.pyplot as plt

HUE = {"base": "#2a78d6", "small": "#eb6834", "tiny": "#1baf7a"}         # categorical slots 1 to 3 of the project palette (colour = model size)
STYLE = {"fp32": "-", "tf32": (0, (1, 1)), "bf16": "--", "fp16": "-."}      # line style = dtype
INK, GRID = "#0b0b0b", "#d9d8d2"
grids_plotted = [g for g in GRIDS if g in fw.grid.values]
assert grids_plotted, "no successful forward timings to plot"
fig, axes = plt.subplots(1, len(grids_plotted), figsize=(4.6 * len(grids_plotted), 3.9), sharey=False, squeeze=False)
for ax, grid in zip(axes[0], grids_plotted):
    for (size, dtype), g in fw[(fw.grid == grid) & (fw.layers == fw[fw.grid == grid].layers.max())].groupby(["size", "dtype"]):
        g = g.sort_values("batch")
        ax.plot(g.batch, g.ms_per_scenario, color=HUE.get(size, "#52514e"), ls=STYLE.get(dtype, "-"), lw=1.6, marker="o", ms=3.5, label=f"{size} {dtype}")
    if grid == "case118_ieee":
        for name, t in BREAKEVEN_MS["paper set (N-1)"].items():
            ax.axhline(t, color="#52514e", lw=0.9, ls=":")
            ax.text(0.01, t, f"break-even, {name}", transform=ax.get_yaxis_transform(), va="bottom", ha="left", fontsize=7, color="#52514e")
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xticks(BATCH_SIZES)
    ax.set_xticklabels([str(b) for b in BATCH_SIZES])
    ax.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    ax.set_xlabel("batch size (scenarios)")
    ax.set_ylabel("forward pass, ms per scenario")
    ax.set_title(grid, loc="left", fontsize=10, color=INK)
    ax.grid(True, color=GRID, lw=0.5)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.legend(frameon=False, fontsize=7)
fig.suptitle(f"GENCO forward pass on {DEVICE_NAME}, full model", fontsize=10, x=0.01, ha="left")
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(OUT / "genco_gpu_timing.png", dpi=150)
plt.show()

## 9. Download

Zips the CSVs, the report, `env.json` and the figure. In the repository, unpack it to `results/gpu_timing/<gpu name>/` (see `notebooks/README.md`).

In [ ]:
slug = "".join(c if c.isalnum() else "_" for c in DEVICE_NAME).strip("_")
zip_path = Path(REPO).parent / f"{'dry_' if DRY_RUN else ''}genco_gpu_timing_{slug}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["genco_gpu_timing.csv", "genco_precision_check.csv", "report.md", "env.json", "genco_gpu_timing.png"]:
        if (OUT / name).exists():
            z.write(OUT / name, f"{slug}/{name}")
    for f in sorted(PARTIAL.glob("*.csv")):
        z.write(f, f"{slug}/partial/{f.name}")
print(zip_path, f"{zip_path.stat().st_size / 1e3:.0f} kB")
for n in zipfile.ZipFile(zip_path).namelist():
    print("  ", n)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:                       # not on Colab, or the browser blocked the download
    print("download from the Files panel (left) instead:", e)

## Troubleshooting

* **No GPU / CUDA not available:** *Runtime > Change runtime type > T4 GPU*, reconnect, run the parameters cell again.
* **`RESTART REQUIRED`:** pip replaced numpy or pandas that were already imported. *Runtime > Restart session*, re-run the parameters cell, skip the upload and install cells.
* **Import errors from `gridfm_graphkit` after the install** (for example a torch API it does not find): the graphkit commit is built for torch 2.10 to 2.13. Run
  `!pip install "torch>=2.10,<2.14" torchvision torchaudio`, restart the session and run the notebook again (the stand-in or a new wheel is picked up automatically).
* **`gridfm_datakit` / `juliacall` messages:** graphkit imports `gridfm-datakit` at module level; the first import may spend a minute on Julia packages. Wait for it.
* **Out of memory:** the configuration is recorded as `OOM` and larger batches of it are skipped. Lower `MAX_BUSES_PER_BATCH` to avoid the attempt, raise it on an A100.
* **bf16 `error` or very slow on a T4:** the T4 has no native bf16 (compute capability 7.5). Use the fp16 rows there.
* **Session disconnected:** re-run the parameters cell, then the cells of sections 1 to 4 (the upload is skipped if `/content/genco` still exists) and section 6; finished models are read from `results/gpu_timing/partial`.
* **Hugging Face rate limits or timeouts:** re-run the data cell; downloaded partitions are kept.